# Практика · Тема 36 · Глибина, стерео і 3D> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)> ⏱ Зошит навчає три крихітні мережі монокулярної глибини. Заміряно:> **близько півтори хвилини** на чотирьох ядрах без відеокарти (сам прогін дав> 77 секунд). Решта клітинок — секунди.Що зробимо:1. Доведемо `assert`ом, що три різні світи дають **побітово однаковий** знімок.2. Побачимо, як друге око цю неоднозначність знімає.3. Порахуємо `Z = F·B/d` і ціну одного пікселя — точно й наближено.4. Заміряємо, що дає й що забирає ширша **база**.5. Подивимось, куди тікає відповідник без **вирівнювання пари**, і полагодимо це   через `cv2.stereoRectify`.6. Порівняємо `StereoBM` і `StereoSGBM` — не «хто кращий», а *де саме* і *якою ціною*.7. Спіймаємо `numDisparities` на тому, що він мовчки переселяє предмет.8. Зберемо **хмару точок** і намалюємо її власним кодом.9. Навчимо монокулярну мережу — і зламаємо її, помноживши світ на 2.5.

In [ ]:
import sys, time, math
import numpy as np
import cv2
import matplotlib.pyplot as plt

print("Python :", sys.version.split()[0])
print("OpenCV :", cv2.__version__)
print("NumPy  :", np.__version__)

# Стала камери на всю тему: фокусна відстань у пікселях і база стереопари в метрах.
FOCAL_PX = 500.0     # скільки пікселів припадає на один метр на відстані один метр
BASELINE_M = 0.10    # відстань між камерами
IMG_W, IMG_H = 320, 240
print(f"\nF = {FOCAL_PX:.0f} px, B = {BASELINE_M:.2f} м, кадр {IMG_W}×{IMG_H}")

## 1 · Три світи дають один знімокКуля радіуса `R` на глибині `Z` дає в кадрі радіус `r = F·R/Z`. Візьмемо три сцени,у яких і радіус, і глибина ростуть у пʼять разів — видимий радіус лишиться тим самим.Яскравість точки на кулі залежить від нахилу поверхні, а нахил — лише від того,наскільки далеко точка від центра **у частках радіуса**. Тому вся картинкавизначається самим лише `r`.

In [ ]:
def render_sphere(radius_px, size=128):
    """Ламбертова куля: яскравість залежить тільки від відстані до центра в частках радіуса."""
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float64)
    center = (size - 1) / 2.0
    u = (xx - center) / radius_px
    v = (yy - center) / radius_px
    dist2 = u * u + v * v
    inside = dist2 <= 1.0
    normal_z = np.sqrt(np.clip(1.0 - dist2, 0, 1))
    # напрямок світла зафіксовано, щоб картинка була однакова у всіх читачів
    shade = np.clip(0.4 * u + 0.5 * v + 0.766 * normal_z, 0, 1)
    out = np.zeros((size, size), np.float64)
    out[inside] = shade[inside]
    return np.round(out * 255).astype(np.uint8)

scenes = [(0.10, 2.0), (0.50, 10.0), (2.50, 50.0)]
images = []
print(f"{'R, м':>6} {'Z, м':>6} {'r, px':>10} {'d, px':>8}")
for radius_m, depth_m in scenes:
    apparent_r = FOCAL_PX * radius_m / depth_m
    disparity = FOCAL_PX * BASELINE_M / depth_m
    images.append(render_sphere(apparent_r))
    print(f"{radius_m:>6.2f} {depth_m:>6.1f} {apparent_r:>10.4f} {disparity:>8.4f}")

А тепер головна перевірка теми. Не «схожі», не «майже однакові» — **рівно нуль**різних пікселів.

In [ ]:
for i in (1, 2):
    different_pixels = int(np.count_nonzero(images[i] != images[0]))
    print(f"різних пікселів між сценою 1 і сценою {i+1}: {different_pixels}")
    assert different_pixels == 0, "сцени мали б збігатися побітово!"
print("\n✅ три світи, що різняться за масштабом у 25 разів, дають побітово однаковий знімок")
print("   масштаб із одного знімка не відновлюється — і це не питання якості моделі")

## 2 · Друге око знімає неоднозначністьПрава камера бачить ту саму кулю зсунутою на `d = F·B/Z` пікселів. Знайдемо цей зсувперебором: для кожного цілого зсуву рахуємо суму квадратів різниці й беремо найменшу.

In [ ]:
def render_sphere_shifted(radius_px, shift_px, size=128):
    """Та сама куля, але центр зсунуто ліворуч — так її бачить права камера."""
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float64)
    center = (size - 1) / 2.0
    u = (xx - center + shift_px) / radius_px
    v = (yy - center) / radius_px
    dist2 = u * u + v * v
    inside = dist2 <= 1.0
    normal_z = np.sqrt(np.clip(1.0 - dist2, 0, 1))
    shade = np.clip(0.4 * u + 0.5 * v + 0.766 * normal_z, 0, 1)
    out = np.zeros((size, size), np.float64)
    out[inside] = shade[inside]
    return np.round(out * 255).astype(np.uint8)

for radius_m, depth_m in scenes:
    apparent_r = FOCAL_PX * radius_m / depth_m
    true_disparity = FOCAL_PX * BASELINE_M / depth_m
    left = render_sphere(apparent_r)
    right = render_sphere_shifted(apparent_r, true_disparity)
    # перебираємо цілі зсуви й шукаємо той, що дає найменшу різницю
    best_cost, best_shift = None, None
    for shift in range(0, 41):
        candidate = np.roll(right, shift, axis=1).astype(np.float64)
        cost = float(((candidate - left) ** 2)[:, 41:].mean())
        if best_cost is None or cost < best_cost:
            best_cost, best_shift = cost, shift
    print(f"Z = {depth_m:>4.0f} м → істинний d = {true_disparity:6.4f} px, знайдено {best_shift} px")
    assert best_shift == round(true_disparity)
print("\n✅ три однакові знімки дали три різні диспаритети: 25 / 5 / 1 px")

## 3 · `Z = F·B/d` і ціна одного пікселяОдин піксель помилки в диспаритеті коштує тим більше, чим далі предмет. Точна ціна —різниця двох значень `Z`; підручникове наближення — `Z²/(F·B)`.

In [ ]:
FB = FOCAL_PX * BASELINE_M       # добуток, який входить у формулу як єдине число
print(f"F·B = {FB:.1f}\n")
print(f"{'Z, м':>6} {'d, px':>8} {'ціна 1 px':>12} {'Z²/(F·B)':>12} {'завищення':>11}")
for depth_m in (1.0, 3.0, 5.0, 9.0, 20.0):
    d = FB / depth_m
    exact = FB / d - FB / (d + 1.0)      # наскільки зміниться Z, якщо d помилиться на 1
    approx = depth_m * depth_m / FB
    label = f"{exact*100:.2f} см" if exact < 1 else f"{exact:.2f} м"
    label2 = f"{approx*100:.2f} см" if approx < 1 else f"{approx:.2f} м"
    print(f"{depth_m:>6.0f} {d:>8.2f} {label:>12} {label2:>12} {approx/exact:>11.3f}")

# перевіряємо, що завищення дорівнює рівно (d + 1) / d
for depth_m in (1.0, 3.0, 5.0, 9.0, 20.0):
    d = FB / depth_m
    ratio = (depth_m * depth_m / FB) / (FB / d - FB / (d + 1.0))
    assert abs(ratio - (d + 1.0) / d) < 1e-9
print("\n✅ наближення завищує рівно в (d + 1) / d разів — тобто на 1/d")

## 4 · База: що дає й що забираєШирша база дає більший диспаритет, отже точнішу глибину. Платить за це **спільне полезору**: для стіни на глибині `Z` зсув між кадрами дорівнює диспаритету, тож спільнимилишаються `1 − F·B/(Z·W)` ширини.Перевіримо обидва боки угоди справжнім зіставленням.

In [ ]:
def textured_plane_pair(disparity_px, seed=0, width=IMG_W, height=IMG_H):
    """Стереопара для однієї плоскої стіни: праве зображення — зсунуте ліве."""
    rng = np.random.default_rng(seed)
    scene = rng.integers(0, 256, size=(height, width + 400), dtype=np.uint8)
    scene = cv2.GaussianBlur(scene, (0, 0), 1.0)
    scene = cv2.normalize(scene, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    left = scene[:, :width].copy()
    map_x = np.tile(np.arange(width, dtype=np.float32) + disparity_px, (height, 1))
    map_y = np.tile(np.arange(height, dtype=np.float32).reshape(-1, 1), (1, width))
    right = cv2.remap(scene, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return left, right

WALL_DEPTH_M = 3.0
print(f"{'B, м':>6} {'d, px':>7} {'nd':>4} {'спільне поле':>13} {'покриття':>9} "
      f"{'похибка d':>10} {'похибка Z':>10}")
for baseline in (0.02, 0.05, 0.10, 0.20, 0.40):
    d_true = FOCAL_PX * baseline / WALL_DEPTH_M
    num_disp = int(math.ceil((d_true + 8) / 16.0)) * 16
    left, right = textured_plane_pair(d_true)
    matcher = cv2.StereoSGBM_create(minDisparity=0, numDisparities=num_disp, blockSize=7,
                                    P1=8*7*7, P2=32*7*7, uniquenessRatio=10,
                                    speckleWindowSize=100, speckleRange=2)
    disp = matcher.compute(left, right).astype(np.float64) / 16.0
    found = disp > 0
    err_px = np.abs(disp[found] - d_true).mean()
    err_m = np.abs(FOCAL_PX * baseline / disp[found] - WALL_DEPTH_M).mean()
    common = 1.0 - FOCAL_PX * baseline / (WALL_DEPTH_M * IMG_W)
    print(f"{baseline:>6.2f} {d_true:>7.2f} {num_disp:>4} {common:>12.1%} "
          f"{found.mean():>9.1%} {err_px:>10.3f} {err_m:>10.3f}")

Похибка **в пікселях** не залежить від бази — це властивість алгоритму зіставлення.А похибка **в метрах** падає у сімнадцять разів. Заплатили за це двадцятьмавідсотками кадру.## 5 · Вирівнювання париДосі ми припускали, що камери стоять ідеально. Подивимось, куди тікає відповідник,якщо праву камеру перекосити. Точки беремо сіткою — щоб числа були відтворюваними.

In [ ]:
K = np.array([[FOCAL_PX, 0, IMG_W/2], [0, FOCAL_PX, IMG_H/2], [0, 0, 1]], float)
DIST = np.zeros(5)                      # дисторсії немає: цікавить сам перекіс

GRID_X = [40, 96, 152, 208, 264]
GRID_Y = [40, 100, 160, 200]
points_3d = []
for i, px in enumerate(GRID_X):
    for j, py in enumerate(GRID_Y):
        depth = 1.5 + 1.5 * ((i + j) % 4)          # глибина циклічно міняється
        points_3d.append([(px - IMG_W/2) * depth / FOCAL_PX,
                          (py - IMG_H/2) * depth / FOCAL_PX, depth])
points_3d = np.array(points_3d)
print("точок у сітці:", len(points_3d))

def rotation(axis, degrees):
    vec = np.zeros(3); vec[axis] = np.deg2rad(degrees)
    return cv2.Rodrigues(vec)[0]

def project(points, R, t):
    rvec, _ = cv2.Rodrigues(R)
    uv, _ = cv2.projectPoints(points, rvec, t, K, DIST)
    return uv.reshape(-1, 2)

uv_left = project(points_3d, np.eye(3), np.zeros(3))
axis_names = {0: "нахил уперед (вісь X)", 1: "поворот убік (вісь Y)", 2: "крен, навколо осі зору"}
print(f"\n{'перекіс правої камери':<26}{'1°':>10}{'2°':>10}{'5°':>10}   (середній розбіг, px)")
for axis in (0, 1, 2):
    row = []
    for degrees in (1.0, 2.0, 5.0):
        uv_right = project(points_3d, rotation(axis, degrees), np.array([-BASELINE_M, 0, 0.0]))
        row.append(np.abs(uv_left[:, 1] - uv_right[:, 1]).mean())
    print(f"{axis_names[axis]:<26}" + "".join(f"{v:>10.2f}" for v in row))

Нахил уперед виносить увесь кадр із рядка, крен розкидає точки віялом, а поворот убікмайже нічого не псує. Тепер полагодимо це `cv2.stereoRectify` і перевіримо`cv2.triangulatePoints`.

In [ ]:
R_skew = rotation(0, 2.0)
T_skew = np.array([[-BASELINE_M], [0], [0.0]])

R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(K, DIST, K, DIST, (IMG_W, IMG_H),
                                            R_skew, T_skew, flags=0, alpha=0)
uv_right = project(points_3d, R_skew, T_skew.ravel())

def rectify_points(uv, R_rect, P_rect):
    src = uv.reshape(-1, 1, 2).astype(np.float64)
    return cv2.undistortPoints(src, K, DIST, R=R_rect, P=P_rect).reshape(-1, 2)

rect_left = rectify_points(uv_left, R1, P1)
rect_right = rectify_points(uv_right, R2, P2)
vertical_gap = np.abs(rect_left[:, 1] - rect_right[:, 1])
print(f"до вирівнювання : середній вертикальний розбіг "
      f"{np.abs(uv_left[:,1]-uv_right[:,1]).mean():.2f} px")
print(f"після вирівнювання: середній вертикальний розбіг {vertical_gap.mean():.2e} px, "
      f"найбільший {vertical_gap.max():.2e} px")

# триангуляція: з двох відбитків повертаємо тривимірні координати
homogeneous = cv2.triangulatePoints(P1, P2, rect_left.T.astype(np.float64),
                                    rect_right.T.astype(np.float64))
xyz_rect = (homogeneous[:3] / homogeneous[3]).T
xyz = (np.linalg.inv(R1) @ xyz_rect.T).T          # повертаємо в систему лівої камери
error_m = np.linalg.norm(xyz - points_3d, axis=1)
print(f"triangulatePoints: середня похибка {error_m.mean():.2e} м, "
      f"найбільша {error_m.max():.2e} м")
print("\n✅ геометрія працює точно — уся справжня похибка стерео живе в пошуку відповідностей")

А тепер найважливіше: скільки коштує **невиправлений** перекіс. Візьмемо готову пару,нахилимо праве зображення на частки градуса й подивимось, що станеться зі`StereoSGBM`.

In [ ]:
def banded_pair(depths_m, flat_box=None, seed=0):
    """Стереопара зі смуг на різних глибинах; flat_box — ділянка без текстури."""
    rng = np.random.default_rng(seed)
    scene = rng.integers(0, 256, size=(IMG_H, IMG_W + 200), dtype=np.uint8)
    scene = cv2.GaussianBlur(scene, (0, 0), 1.0)
    scene = cv2.normalize(scene, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    if flat_box is not None:
        x0, y0, x1, y1 = flat_box
        scene[y0:y1, x0:x1] = 128
    left = scene[:, :IMG_W].copy()
    right = np.zeros_like(left)
    truth = np.zeros((IMG_H, IMG_W), np.float64)
    band_h = IMG_H // len(depths_m)
    for i, depth in enumerate(depths_m):
        d = FOCAL_PX * BASELINE_M / depth
        y0 = i * band_h
        y1 = (i + 1) * band_h if i < len(depths_m) - 1 else IMG_H
        truth[y0:y1] = d
        map_x = np.tile(np.arange(IMG_W, dtype=np.float32) + d, (y1 - y0, 1))
        map_y = np.tile(np.arange(y0, y1, dtype=np.float32).reshape(-1, 1), (1, IMG_W))
        right[y0:y1] = cv2.remap(scene, map_x, map_y, cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_REPLICATE)
    return left, right, truth

BAND_DEPTHS = [1.0, 3.0, 9.0]
FLAT_BOX = (150, 90, 240, 150)
left, right, truth = banded_pair(BAND_DEPTHS, FLAT_BOX)
print("істинні диспаритети смуг:", [f"{FOCAL_PX*BASELINE_M/z:.2f}" for z in BAND_DEPTHS])

NUM_DISP, BLOCK = 64, 7
sgbm = cv2.StereoSGBM_create(minDisparity=0, numDisparities=NUM_DISP, blockSize=BLOCK,
                             P1=8*BLOCK*BLOCK, P2=32*BLOCK*BLOCK, uniquenessRatio=10,
                             speckleWindowSize=100, speckleRange=2)
scored = np.zeros((IMG_H, IMG_W), bool)
scored[12:IMG_H-12, NUM_DISP+10:IMG_W-10] = True   # краї кадру пари не мають

print(f"\n{'нахил':>7} {'зсув рядка':>11} {'покриття':>9} {'похибка d':>10} {'грубих >1px':>12}")
for degrees in (0.0, 0.25, 0.5, 1.0, 2.0):
    R_tilt = cv2.Rodrigues(np.array([np.deg2rad(degrees), 0, 0]))[0]
    H_tilt = K @ R_tilt @ np.linalg.inv(K)
    right_tilted = cv2.warpPerspective(right, H_tilt, (IMG_W, IMG_H),
                                       flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    disp = sgbm.compute(left, right_tilted).astype(np.float64) / 16.0
    ok = scored & (disp > 0)
    err = np.abs(disp - truth)[ok]
    print(f"{degrees:>6.2f}° {FOCAL_PX*np.tan(np.deg2rad(degrees)):>10.2f} "
          f"{ok[scored].mean():>9.1%} {err.mean():>10.3f} {(err > 1).mean():>12.1%}")

Півградуса перекосу — і покриття падає з майже сотні відсотків до сімнадцяти, апохибка росте в пʼятдесят разів. Ось чому калібрування стереопари — не формальність.## 6 · `StereoBM` проти `StereoSGBM`Порівняємо два алгоритми на тій самій парі. Питання не «хто кращий», а **де саме**один виграє і **якою ціною**.

In [ ]:
bm = cv2.StereoBM_create(numDisparities=NUM_DISP, blockSize=BLOCK)
flat_mask = np.zeros((IMG_H, IMG_W), bool)
flat_mask[FLAT_BOX[1]:FLAT_BOX[3], FLAT_BOX[0]:FLAT_BOX[2]] = True

print(f"{'алгоритм':<12}{'час':>9}{'покриття':>10}{'похибка':>10}"
      f"{'грубих':>9}{'гладка ділянка':>16}{'похибка там':>13}")
for name, matcher in (("StereoBM", bm), ("StereoSGBM", sgbm)):
    times = []
    for _ in range(7):        # беремо мінімум із семи: машина може бути зайнята
        t0 = time.perf_counter()
        raw = matcher.compute(left, right)
        times.append(time.perf_counter() - t0)
    disp = raw.astype(np.float64) / 16.0
    ok = scored & (disp > 0)
    err = np.abs(disp - truth)[ok]
    flat_ok = flat_mask & scored & (disp > 0)
    flat_err = np.abs(disp - truth)[flat_ok].mean() if flat_ok.sum() else float("nan")
    print(f"{name:<12}{min(times)*1000:>7.1f}мс{ok[scored].mean():>10.1%}"
          f"{err.mean():>9.3f}px{(err > 1).mean():>9.1%}"
          f"{(flat_ok.sum()/(flat_mask & scored).sum()):>16.1%}{flat_err:>11.3f}px")

Різниця не в точності — вона в останніх двох стовпчиках і в першому: SGBM закриваєгладку ділянку цілком, але коштує майже втричі дорожче.## 7 · `numDisparities` мовчки переселяє предметДіапазон пошуку — це насправді **межа найближчої відстані**, яку камера здатнапобачити. Якщо його не вистачає, предмет не зникає з карти — він переїжджає.

In [ ]:
band_h = IMG_H // 3
print(f"{'nd':>4} {'покриття':>9} | " + " | ".join(f"смуга {z:g} м" for z in BAND_DEPTHS))
for num_disp in (16, 32, 48, 64, 80, 96, 112, 128):
    matcher = cv2.StereoSGBM_create(minDisparity=0, numDisparities=num_disp, blockSize=BLOCK,
                                    P1=8*BLOCK*BLOCK, P2=32*BLOCK*BLOCK, uniquenessRatio=10,
                                    speckleWindowSize=100, speckleRange=2)
    disp = matcher.compute(left, right).astype(np.float64) / 16.0
    cells = []
    for i, depth in enumerate(BAND_DEPTHS):
        y0 = i * band_h
        y1 = (i + 1) * band_h if i < 2 else IMG_H
        band = disp[y0+10:y1-10, 130:IMG_W-20]
        good = band[band > 0]
        if good.size < 50:
            cells.append("   немає значень")
        else:
            median = float(np.median(good))
            cells.append(f"d = {median:5.2f} → Z = {FOCAL_PX*BASELINE_M/median:5.2f} м")
    print(f"{num_disp:>4} {(disp > 0).mean():>8.1%}  | " + " | ".join(cells))
print("\nістинні глибини смуг: 1.00 / 3.00 / 9.00 м")
print("кожні 16 зайвих диспаритетів зʼїдають 16/320 = 5.0 % ширини кадру")

Ближня смуга при `numDisparities=16` дістає **6.90 метра** замість одного — і жодногопопередження. Помилка спадає плавно, тому «поки не виглядає правдоподібно» —найгірший спосіб добирати цей параметр.Далека смуга при цьому скрізь дає 8.51 замість 9.00: знайдений диспаритет 5.88 замість5.56, тобто помилка **0.32 пікселя**. Порівняй із середньою смугою — там помилка0.27 пікселя коштує лише 5 сантиметрів. Це закон Z² усередині одного заміру.## 8 · Хмара точок`cv2.reprojectImageTo3D` переводить карту диспаритету в трійки координат. Спершуперевіримо його на одному числі.

In [ ]:
# Q-матриця: усе, що камера знає про себе, одним масивом 4×4
Q = np.array([[1, 0, 0, -IMG_W/2],
              [0, 1, 0, -IMG_H/2],
              [0, 0, 0, FOCAL_PX],
              [0, 0, 1/BASELINE_M, 0]], float)
single = np.full((1, 1), 25.0, np.float32)
xyz_one = cv2.reprojectImageTo3D(single, Q)
print(f"диспаритет 25 px при F = {FOCAL_PX:.0f}, B = {BASELINE_M:.2f} → Z = {xyz_one[0,0,2]:.4f} м")
assert abs(float(xyz_one[0, 0, 2]) - 2.0) < 1e-6
print("✅ збіглося з першою сценою: куля радіуса 0.10 м на глибині 2 м")

Пакетів для роботи з хмарами (`open3d`, `trimesh`) у нашому середовищі немає — і нетреба. Щоб намалювати хмару, досить тієї самої діркової камери, застосованої узворотний бік: повертаємо точку навколо вертикальної осі й проєктуємо.

In [ ]:
def make_cloud(sigma_px, seed=7):
    """Три пласкі площини на 1, 3 і 9 м; шум додається саме до диспаритету."""
    rng = np.random.default_rng(seed)
    xs = np.linspace(20, 300, 44)
    ys = np.linspace(30, 210, 26)
    grid_x, grid_y = np.meshgrid(xs, ys)
    parts = []
    for plane_i, depth in enumerate(BAND_DEPTHS):
        d_true = FOCAL_PX * BASELINE_M / depth
        d_noisy = d_true + rng.normal(0, sigma_px, grid_x.shape)
        d_noisy = np.clip(d_noisy, 0.2, None)
        Z = FOCAL_PX * BASELINE_M / d_noisy
        X = (grid_x - IMG_W/2) * Z / FOCAL_PX
        Y = (grid_y - IMG_H/2) * Z / FOCAL_PX
        parts.append((X.ravel(), Y.ravel(), Z.ravel(), np.full(X.size, plane_i)))
    return [np.concatenate(a) for a in zip(*parts)]

for sigma in (0.0, 0.30, 1.00):
    X, Y, Z, plane = make_cloud(sigma)
    thick = [Z[plane == i].std() for i in range(3)]
    ratio = f"{thick[2]/thick[0]:.1f}×" if thick[0] > 1e-9 else "—"
    print(f"σ = {sigma:.2f} px → товщина площин: "
          f"{thick[0]:.3f} / {thick[1]:.3f} / {thick[2]:.3f} м, відношення {ratio}")
print("\nтеоретично товщина дорівнює σ·Z²/(F·B): при σ = 0.30 це "
      f"{0.30*1/50:.3f} / {0.30*9/50:.3f} / {0.30*81/50:.3f} м")

In [ ]:
X, Y, Z, plane = make_cloud(0.30)
colors = ["#0f766e", "#c2620f", "#c2185b"]
angle = np.deg2rad(35)
pivot = 4.0
# власна проєкція: поворот навколо вертикальної осі й паралельний вигляд
u = X * np.cos(angle) + (Z - pivot) * np.sin(angle)
v = Y

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for i in range(3):
    sel = plane == i
    axes[0].scatter(X[sel], Z[sel], s=1, c=colors[i], label=f"{BAND_DEPTHS[i]:g} м")
    axes[1].scatter(u[sel], -v[sel], s=1, c=colors[i])
axes[0].set_xlabel("ліворуч-праворуч, м"); axes[0].set_ylabel("глибина, м")
axes[0].set_title("вигляд згори"); axes[0].legend(markerscale=6, fontsize=8)
axes[1].set_title("поворот на 35°"); axes[1].set_aspect("equal"); axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout(); plt.show()
print("далека площина розсипалась у шар завтовшки майже пів метра — при однаковій "
      "похибці в пікселях")

## 9 · Монокулярна глибина: вивчений здогадТепер навчимо мережу оцінювати глибину з одного знімка — і зламаємо її.Сцена: три кулі на стіні. Радіуси беруться з вузького проміжку 0.16-0.24 м, відстані —з проміжку 2-7 м. Мережа передбачає **обернену глибину** (величину 1/Z), як роблятьусі сучасні монокулярні моделі.

In [ ]:
import torch
import torch.nn as nn

torch.set_num_threads(1)          # менше потоків — швидше й відтворювано (див. тему 12)
print("torch:", torch.__version__)

SCENE_SIZE = 48
FOCAL_MONO = 105.0
WALL_M = 9.0

def make_scene(rng, world_scale=1.0):
    """Кулі на стіні. world_scale множить І розміри, І відстані — картинка не змінюється."""
    image = np.zeros((SCENE_SIZE, SCENE_SIZE), np.float32)
    inv_depth = np.full((SCENE_SIZE, SCENE_SIZE), 1.0 / (WALL_M * world_scale), np.float32)
    balls = [(rng.uniform(2.0, 7.0) * world_scale, rng.uniform(0.16, 0.24) * world_scale)
             for _ in range(3)]
    balls.sort(key=lambda pair: -pair[0])            # далекі малюємо першими
    yy, xx = np.mgrid[0:SCENE_SIZE, 0:SCENE_SIZE]
    for depth, radius_m in balls:
        r = FOCAL_MONO * radius_m / depth
        cx = rng.uniform(r + 1, SCENE_SIZE - r - 1)
        cy = rng.uniform(r + 1, SCENE_SIZE - r - 1)
        u = (xx - cx) / r; v = (yy - cy) / r
        dist2 = u * u + v * v
        inside = dist2 <= 1.0
        normal_z = np.sqrt(np.clip(1 - dist2, 0, 1))
        shade = np.clip(0.4 * u + 0.5 * v + 0.766 * normal_z, 0, 1)
        image[inside] = shade[inside] * 0.8 + 0.2
        inv_depth[inside] = 1.0 / depth
    return image, inv_depth

def make_dataset(count, seed, world_scale=1.0):
    rng = np.random.default_rng(seed)
    X = np.zeros((count, 1, SCENE_SIZE, SCENE_SIZE), np.float32)
    Y = np.zeros((count, 1, SCENE_SIZE, SCENE_SIZE), np.float32)
    for i in range(count):
        image, inv_depth = make_scene(rng, world_scale)
        X[i, 0] = image; Y[i, 0] = inv_depth
    return torch.from_numpy(X), torch.from_numpy(Y)

train_x, train_y = make_dataset(600, seed=1)
test_x, test_y = make_dataset(200, seed=2)
big_x, big_y = make_dataset(200, seed=2, world_scale=2.5)

same = bool(torch.equal(test_x, big_x))
print("картинки світу ×2.5 збігаються з рідними побітово:", same)
assert same, "світ помножився, а знімки мали б лишитись тими самими!"
print("✅ той самий доказ, що й у першому розділі, але вже на цілому датасеті")

# передбачаємо обернену глибину, помножену на 10 — щоб значення були зручного масштабу
train_y, test_y, big_y = train_y * 10, test_y * 10, big_y * 10

In [ ]:
class TinyDepthNet(nn.Module):
    """Крихітний енкодер-декодер: три стискання, три розтягання."""
    def __init__(self):
        super().__init__()
        def block(a, b):
            return [nn.Conv2d(a, b, 3, padding=1), nn.ReLU()]
        self.body = nn.Sequential(
            *block(1, 8), nn.MaxPool2d(2),
            *block(8, 16), nn.MaxPool2d(2),
            *block(16, 24), nn.MaxPool2d(2),
            *block(24, 24),
            nn.Upsample(scale_factor=2), *block(24, 16),
            nn.Upsample(scale_factor=2), *block(16, 8),
            nn.Upsample(scale_factor=2), nn.Conv2d(8, 1, 3, padding=1))

    def forward(self, x):
        return self.body(x)

print("параметрів:", sum(p.numel() for p in TinyDepthNet().parameters()))

results = []
for seed in (0, 1, 2):
    torch.manual_seed(seed)
    net = TinyDepthNet()
    optimizer = torch.optim.AdamW(net.parameters(), lr=4e-3)
    started = time.perf_counter()
    for epoch in range(24):
        order = torch.randperm(len(train_x))
        for i in range(0, len(train_x), 32):
            batch = order[i:i+32]
            loss = ((net(train_x[batch]) - train_y[batch]) ** 2).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    elapsed = time.perf_counter() - started

    with torch.no_grad():
        pred = net(test_x).clamp(min=0.05)
    depth_pred = 10.0 / pred
    depth_true = 10.0 / test_y
    depth_big = 10.0 / big_y
    mae = (depth_pred - depth_true).abs().mean().item()
    delta1 = (torch.max(depth_pred/depth_true, depth_true/depth_pred) < 1.25).float().mean().item()
    mae_big = (depth_pred - depth_big).abs().mean().item()
    # підбираємо ОДИН множник на всі пікселі всіх сцен — так міряють монокулярну глибину
    scale = torch.median(big_y / pred).item()
    mae_scaled = ((10.0 / (pred * scale)) - depth_big).abs().mean().item()
    results.append((seed, elapsed, mae, delta1, mae_big, mae_scaled, scale))
    print(f"зерно {seed}: {elapsed:5.1f} с | MAE {mae:.3f} м | δ<1.25 {delta1:.4f} | "
          f"світ ×2.5: MAE {mae_big:.3f} м → після одного множника {mae_scaled:.3f} м "
          f"(множник {scale:.3f})")

In [ ]:
maes = [r[2] for r in results]
bigs = [r[4] for r in results]
scales = [r[6] for r in results]
print(f"розкид MAE на рідному світі : {min(maes):.3f} … {max(maes):.3f} м")
print(f"розкид MAE на світі ×2.5    : {min(bigs):.3f} … {max(bigs):.3f} м")
print(f"підібраний множник          : {min(scales):.3f} … {max(scales):.3f}  (1/2.5 = 0.400)")
print()
print("Розкид по зернах на рідному світі майже вдвічі — на таких числах не можна")
print("заявляти про поліпшення на десять відсотків. Зате ефект ×2.5 у десятки разів")
print("більший за розкид, тож він справжній.")
print()
print("Мережа не помилилась у формі сцени — вона не знала ОДНОГО числа, і цього")
print("числа не було в зображенні. Саме тому в статтях про монокулярну глибину")
print("передбачення перед порівнянням домножують на множник, підібраний за")
print("еталонною розміткою.")

In [ ]:
# подивимось на одну сцену: знімок, істинна глибина й передбачення
with torch.no_grad():
    pred = net(test_x[:1]).clamp(min=0.05)
sample = 0
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(test_x[sample, 0], cmap="gray"); axes[0].set_title("знімок")
im1 = axes[1].imshow(10.0 / test_y[sample, 0], cmap="viridis"); axes[1].set_title("істинна глибина, м")
im2 = axes[2].imshow((10.0 / pred[0, 0]).numpy(), cmap="viridis"); axes[2].set_title("передбачення, м")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(im1, ax=axes[1], fraction=0.046); plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout(); plt.show()
print("Форма сцени вгадана; абсолютні метри тримаються лише на тому, що кулі в")
print("навчальному наборі були приблизно однакового розміру.")

## Завдання### 🟢 Рівень 1Повтори перший розділ із **кубом** замість кулі: намалюй квадрат зі стороною`s = F·S/Z` для трьох пар (S, Z), що дають однакову сторону в пікселях, і доведи`assert`ом, що зображення збігаються побітово.### 🟡 Рівень 2Візьми таблицю з розділу 4 і додай стовпчик «на якій відстані диспаритет падає доодного пікселя» (це рівно `F·B`). Побудуй графік: по одній осі база, по другій — цямежа. Поясни словами, чому крива пряма.### 🔴 Рівень 3Заміни в розділі 7 `StereoSGBM` на `StereoBM` і повтори перебір `numDisparities`.Чи так само тихо переселяється ближня смуга? Порівняй, як два алгоритми поводятьсятам, де правильної відповіді в діапазоні пошуку немає: хто частіше зізнається, що незнає?